<a href="https://colab.research.google.com/github/Riccardo-27/DL-Quantum/blob/dev-riccardo/notebooks/02_windowing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 – Generazione Finestre Temporali, Super-Risoluzione e Verifiche Anti-Leakage

Questo notebook implementa la **Fase 2 (Step S3)** della pipeline di previsione della dinamica quantistica:

1. **Finestrature Standard di Progetto (`short` e `long`)**:
   - Generazione di finestre a frequenza nativa ($\Delta t = 0.02$) con stride fisso $\Delta s = 10$.
   - Configurazione `short`: $T_{in} = 10$ ($0.20$ unità temporali) $	o T_{out} = 40$ ($0.80$ unità temporali).
   - Configurazione `long`: $T_{in} = 20$ ($0.40$ unità temporali) $	o T_{out} = 40$ ($0.80$ unità temporali).
2. **Griglia $T_{in} \in \{5, 10, 20, 40\}$ per la Sfida 1 (Finestra Minima)**:
   - Valutazione dell'orizzonte di osservazione minimo ottimale.
   - Allineamento temporale dei target (`align_t_in=40`): assicura che finestre con $T_{in}$ diversi valutino esattamente gli stessi istanti futuri $Y$.
3. **Generazione Finestre per la Sfida 2 (Super-Risoluzione Temporale)**:
   - Subsampling temporale $	imes 2$: input $X_{SR}$ a frequenza dimezzata ($\Delta t = 0.04$, 20 punti su span di 39 step) per ricostruire la dinamica continua nativa a $\Delta t = 0.02$ su $T_{out} = 40$.
4. **Input per Decoder e Teacher Forcing**:
   - Generazione di $Y_{dec}$ shiftato a destra per addestramento seq2seq e Transformer.
5. **Verifiche di Integrità e Assoluta Assenza di Data Leakage**:
   - Isolamento formale per traiettoria: nessuna finestra scavalca il confine tra due traiettorie.
   - Controllo di disgiunzione assoluta tra Train (280), Val (60) e Test (60).
6. **Esportazione Grafici Standardizzati**:
   - Curve di contesto e target, confronto super-risoluzione ed evidenza di allineamento salvati in `figures/02_windowing/` in formato JPEG ad alta risoluzione ($\ge 300$ dpi).


## 0. Setup dell'Ambiente, Sincronizzazione Git e Determinismo

Configuriamo l'ambiente di esecuzione (Google Colab vs locale), inizializziamo i seed deterministici (`SEED = 42`) e ci allineiamo sul branch `dev-riccardo`.


In [ ]:
# =============================================================================
# 0. Setup Ambiente, Sincronizzazione Git e Riproducibilità
# =============================================================================
import os
import sys
import time
from pathlib import Path

# Rilevamento automatico dell'ambiente di esecuzione (Google Colab vs Locale)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_REPO = Path("/content/drive/MyDrive/DL-Quantum")
    LOCAL_REPO = Path("/content/DL-Quantum")

    # Risoluzione robusta del repository tramite verifica del file sentinella src/seeding.py
    if (DRIVE_REPO / "src" / "seeding.py").exists():
        REPO_PATH = DRIVE_REPO
        print("Repository trovato su Google Drive. Sincronizzazione git...")
        !git -C "{REPO_PATH}" fetch origin
        !git -C "{REPO_PATH}" checkout dev-riccardo
        !git -C "{REPO_PATH}" pull origin dev-riccardo
    elif (LOCAL_REPO / "src" / "seeding.py").exists():
        REPO_PATH = LOCAL_REPO
        print("Repository trovato in /content. Sincronizzazione git...")
        !git -C "{REPO_PATH}" fetch origin
        !git -C "{REPO_PATH}" checkout dev-riccardo
        !git -C "{REPO_PATH}" pull origin dev-riccardo
    else:
        print("Cartella DL-Quantum: clonazione o allineamento da GitHub...")
        if DRIVE_REPO.exists():
            !git -C /content/drive/MyDrive/DL-Quantum init
            !git -C /content/drive/MyDrive/DL-Quantum remote add origin https://github.com/Riccardo-27/DL-Quantum.git
            !git -C /content/drive/MyDrive/DL-Quantum fetch origin
            !git -C /content/drive/MyDrive/DL-Quantum checkout -f dev-riccardo
            REPO_PATH = DRIVE_REPO
        else:
            !git clone -b dev-riccardo https://github.com/Riccardo-27/DL-Quantum.git /content/drive/MyDrive/DL-Quantum
            REPO_PATH = DRIVE_REPO
else:
    # Risoluzione per ambiente locale (notebook vive nella sottocartella notebooks/)
    REPO_PATH = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

# Configurazione del path di importazione dei moduli src/
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Variabili d'ambiente di progetto
os.environ.setdefault("DLQ_AUTHOR", "riccardo")
os.environ.setdefault("DLQ_ROOT", str(REPO_PATH))

# Determinismo rigoroso
from src.seeding import set_seeds
SEED = 42
set_seeds(SEED)

print(f"Radice progetto: {REPO_PATH} | Ambiente Colab: {IN_COLAB}")


In [ ]:
# =============================================================================
# Import Moduli di Progetto e Configurazione Percorsi
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (
    DT, N_FEATURES, N_POINTS, N_QUBITS, N_TRAJ, get_paths, make_config
)
from src.data import feature_names, load_split, load_trajectories
from src.plots import plot_window, save_figure
from src.windowing import (
    check_no_leakage, decoder_inputs, make_sr_windows,
    make_windows, target_time_index, window_starts
)

CFG = make_config()
PATHS = get_paths()
NAMES = feature_names()

print("Configurazione predefinita caricata:")
print(f"  Orizzonte futuro (T_out)    : {CFG['t_out']} passi ({CFG['t_out']*DT:.2f} unita' fisiche)")
print(f"  Passo di scorrimento (stride): {CFG['stride']} passi ({CFG['stride']*DT:.2f} unita' fisiche)")
print(f"  Finestre standard           : {CFG['windows']}")
print(f"  Griglia Sfida 1 (T_in_grid) : {CFG['t_in_grid']}")
print(f"  Fattore Super-Risoluzione   : x{CFG['sr_factor']}")


## 1. Caricamento Dataset e Split Congelato

Carichiamo il tensore strutturato 3D `(400, 1001, 55)` dalla cache binaria `trajectories.npy` (88 MB) generata nel Notebook 01 e recuperiamo la partizione congelata da `split_indices.npz` (280 Train, 60 Val, 60 Test).


In [ ]:
# =============================================================================
# 1. Caricamento Dati e Indici di Split Congelati
# =============================================================================
t0 = time.time()
data = load_trajectories()
load_time = time.time() - t0

split_path = PATHS["results"] / "split_indices.npz"
assert split_path.exists(), f"Split indices non trovati in {split_path}! Esegui prima 01_data.ipynb."
split = load_split(split_path)

print(f"Dataset caricato in {load_time:.2f} s | Shape tensore: {data.shape} (float32)")
print(f"Partizioni: Train={len(split['train'])}, Val={len(split['val'])}, Test={len(split['test'])}")
assert data.shape == (N_TRAJ, N_POINTS, N_FEATURES), "Shape dataset non conforme!"
assert len(split["train"]) + len(split["val"]) + len(split["test"]) == N_TRAJ, "Partizione incompleta!"
print("Integrita' dati e partizioni: verificata con successo ✔")


## 2. Finestrature Standard di Progetto (`short` e `long`)

Tutti i modelli del progetto (RNN e Transformer) condivideranno esattamente le medesime finestrature standard:
* **`short`**: $T_{in} = 10 	o T_{out} = 40$ (span = 50 step, $	ext{stride} = 10$).
* **`long`**: $T_{in} = 20 	o T_{out} = 40$ (span = 60 step, $	ext{stride} = 10$).

L'orizzonte futuro fisso $T_{out} = 40$ corrisponde a $\Delta t 	imes 40 = 0.80$ unità di tempo fisico, consentendo un confronto imparziale tra i due diversi contesti di memoria passata.


In [ ]:
# =============================================================================
# 2. Generazione Finestre Standard (short & long)
# =============================================================================
standard_windows = {}

for name, t_in in CFG["windows"].items():
    t_out = CFG["t_out"]
    stride = CFG["stride"]
    span = t_in + t_out
    
    t_start = time.time()
    X_tr, Y_tr, meta_tr = make_windows(data, split["train"], t_in=t_in, t_out=t_out, stride=stride)
    X_va, Y_va, meta_va = make_windows(data, split["val"],   t_in=t_in, t_out=t_out, stride=stride)
    X_te, Y_te, meta_te = make_windows(data, split["test"],  t_in=t_in, t_out=t_out, stride=stride)
    elapsed = time.time() - t_start
    
    # Memoria RAM occupata dai tensori
    total_bytes = (X_tr.nbytes + Y_tr.nbytes + X_va.nbytes + Y_va.nbytes + X_te.nbytes + Y_te.nbytes)
    total_mb = total_bytes / (1024 ** 2)
    
    standard_windows[name] = {
        "train": (X_tr, Y_tr, meta_tr),
        "val": (X_va, Y_va, meta_va),
        "test": (X_te, Y_te, meta_te),
        "t_in": t_in,
        "t_out": t_out,
        "stride": stride,
        "span": span,
        "total_mb": total_mb
    }
    
    print(f"--- Finestra '{name}' (T_in={t_in}, T_out={t_out}, span={span}, stride={stride}) ---")
    print(f"  Train: X={X_tr.shape}, Y={Y_tr.shape} ({len(X_tr)} finestre)")
    print(f"  Val  : X={X_va.shape}, Y={Y_va.shape} ({len(X_va)} finestre)")
    print(f"  Test : X={X_te.shape}, Y={Y_te.shape} ({len(X_te)} finestre)")
    print(f"  RAM totale: {total_mb:.1f} MB | Tempo estrazione: {elapsed:.3f} s\n")


## 3. Esplorazione della Griglia $T_{in}$ per la Sfida 1 (Finestra Minima Ottimale)

La **Sfida 1** richiede di determinare la *finestra temporale di input più corta in grado di garantire previsioni accurate* sulla dinamica quantistica.
Testiamo la griglia $T_{in} \in \{5, 10, 20, 40\}$ con $T_{out} = 40$ fisso.

### Allineamento Temporale dei Target (`align_t_in = 40`)
Se estraessimo le finestre senza allineamento, una finestra con $T_{in}=5$ inizierebbe a prevedere a $t = 0.10$, mentre una con $T_{in}=40$ a $t = 0.80$: i modelli verrebbero valutati su segmenti di traiettoria disallineati!
Con `align_t_in = 40`, trasliamo le partenze delle finestre più corte in modo che **tutti i modelli valutino esattamente gli stessi target futuri $Y$**, consentendo un confronto scientificamente rigoroso e privo di bias temporali.


In [ ]:
# =============================================================================
# 3. Griglia Tin per la Sfida 1 (con e senza allineamento target)
# =============================================================================
tin_grid = CFG["t_in_grid"]  # [5, 10, 20, 40]
t_out = CFG["t_out"]
stride = CFG["stride"]

grid_results = []
aligned_targets = {}

for t_in in tin_grid:
    # 1. Estrazione standard (non allineata)
    X_std, Y_std, meta_std = make_windows(data, split["val"], t_in=t_in, t_out=t_out, stride=stride)
    
    # 2. Estrazione allineata (align_t_in = 40)
    X_aln, Y_aln, meta_aln = make_windows(
        data, split["val"], t_in=t_in, t_out=t_out, stride=stride, align_t_in=max(tin_grid)
    )
    aligned_targets[t_in] = Y_aln
    
    grid_results.append({
        "T_in": t_in,
        "T_out": t_out,
        "Span Standard": t_in + t_out,
        "Finestre Standard": len(X_std),
        "Finestre Allineate": len(X_aln),
        "Primo t target Standard": f"{t_in * DT:.2f} s",
        "Primo t target Allineato": f"{max(tin_grid) * DT:.2f} s",
        "RAM Val (MB)": f"{(X_aln.nbytes + Y_aln.nbytes)/(1024**2):.2f}"
    })

df_grid = pd.DataFrame(grid_results)
display(df_grid)

# Verifica di identità bit-a-bit dei target allineati tra tutte le finestre Tin
ref_target = aligned_targets[max(tin_grid)]
for t_in in tin_grid:
    np.testing.assert_array_equal(
        aligned_targets[t_in], ref_target,
        err_msg=f"Discrepanza nei target allineati per T_in={t_in}!"
    )

print("\nVerifica allineamento target per la Sfida 1: SUPERATA ✔")
print("Tutte le finestre Tin condividono esattamente gli stessi", len(ref_target), "target futuri!")


## 4. Generazione Finestre per la Sfida 2: Super-Risoluzione Temporale

Nella **Sfida 2**, il modello deve ricostruire la dinamica continua a frequenza nativa ($\Delta t = 0.02$) su $T_{out} = 40$ ricevendo in ingresso un segnale **sottocampionato** con passo dimezzato ($2 	imes \Delta t = 0.04$).

* **Input ($X_{SR}$)**: $T_{in} = 20$ punti osservati a intervalli di $\Delta t = 0.04$, che coprono uno span continuo di $(20 - 1) 	imes 2 + 1 = 39$ passi temporali nativi.
* **Target ($Y_{SR}$)**: $T_{out} = 40$ passi consecutivi a risoluzione piena $\Delta t = 0.02$.
* **Ground Truth Denso ($X_{dense}$)**: tutti i 39 punti nativi del contesto, conservati per validare e visualizzare il sottocampionamento.


In [ ]:
# =============================================================================
# 4. Finestre per Super-Risoluzione (Sfida 2)
# =============================================================================
sr_factor = CFG["sr_factor"]  # 2
sr_t_in = CFG["sr_t_in"]      # 20
t_out = CFG["t_out"]
stride = CFG["stride"]

# Generazione con restituzione dell'input denso continuo
X_sr_tr, Y_sr_tr, meta_sr_tr, Xd_tr = make_sr_windows(
    data, split["train"], t_in=sr_t_in, t_out=t_out, stride=stride,
    factor=sr_factor, return_dense_input=True
)
X_sr_va, Y_sr_va, meta_sr_va, Xd_va = make_sr_windows(
    data, split["val"], t_in=sr_t_in, t_out=t_out, stride=stride,
    factor=sr_factor, return_dense_input=True
)
X_sr_te, Y_sr_te, meta_sr_te, Xd_te = make_sr_windows(
    data, split["test"], t_in=sr_t_in, t_out=t_out, stride=stride,
    factor=sr_factor, return_dense_input=True
)

print(f"Super-Risoluzione (fattore x{sr_factor}):")
print(f"  Train: X_sr={X_sr_tr.shape} (osservato), X_dense={Xd_tr.shape} (denso), Y={Y_sr_tr.shape}")
print(f"  Val  : X_sr={X_sr_va.shape}, Y={Y_sr_va.shape}")
print(f"  Test : X_sr={X_sr_te.shape}, Y={Y_sr_te.shape}")

# Verifica formale: ogni punto di X_sr corrisponde esattamente al punto a passo doppio del denso
np.testing.assert_array_equal(X_sr_tr, Xd_tr[:, ::sr_factor], err_msg="Errore nel sottocampionamento!")
print("\nVerifica coerenza temporale del sottocampionamento: superata ✔")


## 5. Input per il Decoder e Teacher Forcing

Per i modelli sequenza-a-sequenza (RNN encoder-decoder e Transformer), l'addestramento ibrido richiede il target reale shiftato a destra di un passo temporale ($Y_{dec}$):
* Al primo istante $t=0$, il decoder riceve l'ultimo valore osservato dall'encoder: $Y_{dec}[:, 0] = X[:, -1]$.
* Ai passi successivi $t \ge 1$, riceve il target reale al passo precedente: $Y_{dec}[:, 1:] = Y[:, :-1]$.

Verifichiamo che la funzione `decoder_inputs` rispetti questa formulazione.


In [ ]:
# =============================================================================
# 5. Generazione e Verifica degli Input del Decoder (Teacher Forcing)
# =============================================================================
X_sample, Y_sample, _ = standard_windows["short"]["train"]
dec_in_sample = decoder_inputs(X_sample[:100], Y_sample[:100])

print("Shape verifica Teacher Forcing:")
print(f"  Input encoder X: {X_sample[:100].shape}")
print(f"  Target reale Y : {Y_sample[:100].shape}")
print(f"  Input decoder  : {dec_in_sample.shape}")

# Verifiche formali di correttezza dello shift temporale
np.testing.assert_array_equal(dec_in_sample[:, 0], X_sample[:100, -1],
                              err_msg="Il primo step del decoder deve coincidere con l'ultimo step dell'encoder!")
np.testing.assert_array_equal(dec_in_sample[:, 1:], Y_sample[:100, :-1],
                              err_msg="I passi successivi devono essere identici a Y shiftato di 1!")
print("\nVerifica allineamento Teacher Forcing: SUPERATA ✔")


## 6. Verifiche di Integrità e Assoluta Assenza di Data Leakage

Eseguiamo tre controlli formali matematici per garantire la totale correttezza e assenza di leakage:
1. **Confinamento intra-traiettoria**: nessuna finestra supera l'estremo finale $N_{points} = 1001$.
2. **Disgiunzione assoluta**: nessuna traiettoria appartiene contemporaneamente a Train, Val o Test.
3. **Coerenza dei metadati**: verifica con la funzione di progetto `check_no_leakage()`.


In [ ]:
# =============================================================================
# 6. Verifiche Formali Anti-Leakage
# =============================================================================
for name, w_dict in standard_windows.items():
    span = w_dict["span"]
    metas = {
        "train": w_dict["train"][2],
        "val": w_dict["val"][2],
        "test": w_dict["test"][2]
    }
    
    # 1. Nessuna finestra attraversa o eccede i 1001 punti della traiettoria
    for split_name, meta in metas.items():
        max_end = np.max(meta[:, 1] + span)
        assert max_end <= N_POINTS, f"Finestra oltre il limite ({max_end} > {N_POINTS}) in {split_name}!"
    
    # 2. Controllo formale disgiunzione mediante check_no_leakage
    check_no_leakage(metas)
    print(f"Finestra '{name:5s}': isolamento intra-traiettoria e zero leakage verificati ✔")

# Verifica analoga sui metadati di Super-Risoluzione
check_no_leakage({"train": meta_sr_tr, "val": meta_sr_va, "test": meta_sr_te})
print("Super-Risoluzione: isolamento e zero leakage verificati ✔")


## 7. Visualizzazione Grafica Standardizzata (JPEG $\ge 300$ dpi)

Generiamo e salviamo in `figures/02_windowing/` le figure ufficiali ad alta risoluzione:
1. `s3_window_sample_short.jpeg`: finestra `short` per magnetizzazione ($m_1$) e correlazione ($c_{1,2}$).
2. `s3_window_sample_long.jpeg`: finestra `long` corrispondente.
3. `s3_window_super_resolution.jpeg`: visualizzazione intuitiva della super-risoluzione temporale.
4. `s3_tin_grid_comparison.jpeg`: confronto delle 4 finestre $T_{in}$ con target allineati per la Sfida 1.


In [ ]:
# =============================================================================
# 7. Generazione Grafici Ufficiali (figures/02_windowing/)
# =============================================================================
fig_dir = PATHS["figures"] / "02_windowing"
fig_dir.mkdir(parents=True, exist_ok=True)

# --- 1. Finestra Corta (short: Tin=10, Tout=40) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_window(standard_windows["short"]["train"][0][0], standard_windows["short"]["train"][1][0],
            feature=0, dt=DT, ax=axes[0], title="Finestra Short (T_in=10, T_out=40) - Magnetizzazione m1")
plot_window(standard_windows["short"]["train"][0][0], standard_windows["short"]["train"][1][0],
            feature=10, dt=DT, ax=axes[1], title="Finestra Short (T_in=10, T_out=40) - Correlazione c1,2")
plt.tight_layout()
p1 = save_figure(fig, "s3_window_sample_short.jpeg", subdir="02_windowing")
plt.show()
print("Salvata:", p1)

# --- 2. Finestra Lunga (long: Tin=20, Tout=40) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_window(standard_windows["long"]["train"][0][0], standard_windows["long"]["train"][1][0],
            feature=0, dt=DT, ax=axes[0], title="Finestra Long (T_in=20, T_out=40) - Magnetizzazione m1")
plot_window(standard_windows["long"]["train"][0][0], standard_windows["long"]["train"][1][0],
            feature=10, dt=DT, ax=axes[1], title="Finestra Long (T_in=20, T_out=40) - Correlazione c1,2")
plt.tight_layout()
p2 = save_figure(fig, "s3_window_sample_long.jpeg", subdir="02_windowing")
plt.show()
print("Salvata:", p2)

# --- 3. Super-Risoluzione Temporale (Sfida 2) ---
fig, ax = plt.subplots(figsize=(10, 4))
feat_idx = 0  # m1
# Coordinate temporali per il plotting
t_obs_dense = np.arange(Xd_tr.shape[1]) * DT
t_obs_sparse = np.arange(X_sr_tr.shape[1]) * (DT * sr_factor)
t_future = (Xd_tr.shape[1] + np.arange(Y_sr_tr.shape[1])) * DT

ax.plot(t_obs_dense, Xd_tr[0, :, feat_idx], color="gray", ls="--", lw=1.2, label="Dinamica continua (ground truth denso)")
ax.scatter(t_obs_sparse, X_sr_tr[0, :, feat_idx], color="#1f77b4", s=35, zorder=3, label=f"Input osservato radi (x{sr_factor}, dt=0.04)")
ax.plot(t_future, Y_sr_tr[0, :, feat_idx], color="#d62728", lw=1.8, label="Target da prevedere (dt=0.02)")
ax.axvline(t_obs_dense[-1], color="black", ls=":", lw=1)
ax.set_xlabel("Tempo (t)")
ax.set_ylabel(NAMES[feat_idx])
ax.set_title(f"Sfida 2: Super-Risoluzione Temporale - {NAMES[feat_idx]}")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
p3 = save_figure(fig, "s3_window_super_resolution.jpeg", subdir="02_windowing")
plt.show()
print("Salvata:", p3)

# --- 4. Confronto Allineamento Griglia Tin (Sfida 1) ---
fig, axes = plt.subplots(len(tin_grid), 1, figsize=(11, 8), sharex=True)
for i, t_in in enumerate(tin_grid):
    X_aln, Y_aln, meta_aln = make_windows(data, split["val"][:1], t_in=t_in, t_out=t_out, stride=stride, align_t_in=max(tin_grid))
    t_in_coords = (max(tin_grid) - t_in + np.arange(t_in)) * DT
    t_out_coords = (max(tin_grid) + np.arange(t_out)) * DT
    
    axes[i].plot(t_in_coords, X_aln[0, :, 0], "o-", ms=3, label=f"Input T_in={t_in} ({t_in*DT:.2f} s)")
    axes[i].plot(t_out_coords, Y_aln[0, :, 0], "-", color="#2ca02c", label=f"Target T_out={t_out} (condiviso)")
    axes[i].axvline(max(tin_grid) * DT, color="gray", ls=":", lw=1)
    axes[i].set_ylabel("m1")
    axes[i].legend(loc="upper left", fontsize=8)
    axes[i].set_title(f"Contesto T_in = {t_in} step", fontsize=10)

axes[-1].set_xlabel("Tempo t (relativo)")
plt.suptitle("Sfida 1: Allineamento Temporale dei Target tra diverse finestre T_in", fontsize=12, y=1.01)
plt.tight_layout()
p4 = save_figure(fig, "s3_tin_grid_comparison.jpeg", subdir="02_windowing")
plt.show()
print("Salvata:", p4)


## 8. Benchmark delle Prestazioni I/O & In-Memory Pipeline

Confrontiamo la velocità di generazione delle finestre con l'approccio vettorizzato `sliding_window_view` rispetto al salvataggio persistente su disco.
Poiché l'estrazione dell'intero set di addestramento richiede **meno di 0.4 secondi**, la generazione dinamica in RAM all'avvio degli esperimenti è ampiamente preferibile, evitando sprechi di memoria su disco (centinaia di MB ridondanti per ciascuna combinazione).


In [ ]:
# =============================================================================
# 8. Benchmark Prestazioni Estrazione in Memoria
# =============================================================================
trials = 5
durations = []

for _ in range(trials):
    t0 = time.time()
    _ = make_windows(data, split["train"], t_in=10, t_out=40, stride=10)
    durations.append(time.time() - t0)

mean_dur = np.mean(durations)
std_dur = np.std(durations)
throughput = len(split["train"]) / mean_dur

print(f"Benchmark make_windows (280 traiettorie x 1001 punti -> 26.880 finestre):")
print(f"  Tempo medio : {mean_dur*1000:.1f} +- {std_dur*1000:.1f} ms")
print(f"  Throughput  : {throughput:.1f} traiettorie/secondo")
print(f"  Throughput  : {len(standard_windows['short']['train'][0]) / mean_dur:.0f} finestre/secondo")
print("\nConclusione: generazione in RAM istantanea, nessun salvataggio su disco necessario ✔")


## 9. Checklist Finale di Validazione (Step S3)

Verifichiamo che tutti i requisiti ingegneristici della Fase 2 siano formalmente soddisfatti prima di procedere con l'addestramento dei modelli.


In [ ]:
# =============================================================================
# 9. Checklist Finale di Validazione
# =============================================================================
checklist = {
    "Shape standard 'short' coerente (26880, 10, 55)": standard_windows["short"]["train"][0].shape == (26880, 10, 55),
    "Shape standard 'long' coerente (26600, 20, 55)": standard_windows["long"]["train"][0].shape == (26600, 20, 55),
    "Orizzonte di previsione T_out = 40 (0.80 s) fisso": all(w["t_out"] == 40 for w in standard_windows.values()),
    "Target allineati per la Sfida 1 (align_t_in=40)": all(np.array_equal(aligned_targets[k], ref_target) for k in tin_grid),
    "Finestre Super-Risoluzione x2 generate correttamente": X_sr_tr.shape == (26040, 20, 55),
    "Allineamento Teacher Forcing decoder_inputs verificato": np.array_equal(dec_in_sample[:, 0], X_sample[:100, -1]),
    "Assenza totale di Data Leakage (check_no_leakage)": True,
    "4 Figure JPEG ad alta risoluzione (>= 300 dpi) esportate": all(p.exists() for p in (p1, p2, p3, p4)),
}

for k, v in checklist.items():
    print(f"{'✔' if v else '✘'} {k}")

all_passed = all(checklist.values())
print("\n" + ("=" * 50))
print("VALIDAZIONE FINESTRE TEMPORALI:", "COMPLETATA CON SUCCESSO ✔" if all_passed else "ANOMALIE RILEVATE ✘")
print("=" * 50)


## 10. Sincronizzazione dei Risultati e Push su GitHub

Questa cella finale consente di salvare le figure esportate e le modifiche del notebook direttamente su GitHub sul branch `dev-riccardo`.


In [ ]:
# =============================================================================
# 10. Sincronizzazione dei Risultati e Push su GitHub
# =============================================================================
import os
from pathlib import Path

target_repo = Path("/content/drive/MyDrive/DL-Quantum") if Path("/content/drive/MyDrive/DL-Quantum").exists() else REPO_PATH

# 1. Rileva il branch attuale
res_branch = !git -C "{target_repo}" branch --show-current
current_branch = res_branch[0].strip() if res_branch and res_branch[0].strip() else "dev-riccardo"

print(f"--- SINCRONIZZAZIONE SUL BRANCH: {current_branch} ---")
!git -C "{target_repo}" status -s

# 2. Configura l'identità Git di chi esegue solo se non ancora impostata
existing_user = !git -C "{target_repo}" config user.name
if not existing_user or not existing_user[0].strip():
    !git -C "{target_repo}" config user.name "Riccardo"
    !git -C "{target_repo}" config user.email "lavoratoriccardo@gmail.com"

# 3. Aggiunge le figure e il notebook al tracking
!git -C "{target_repo}" add figures/02_windowing/ notebooks/02_windowing.ipynb src/
commit_msg = f"feat(02_windowing): finestre temporali, super-risoluzione e verifiche anti-leakage [{current_branch}]"
!git -C "{target_repo}" commit -m "{commit_msg}"

# 4. Push su GitHub tramite Secret Colab personale (GH_TOKEN o GITHUB_TOKEN)
try:
    from google.colab import userdata
    token = None
    for key in ("GH_TOKEN", "GITHUB_TOKEN"):
        try:
            token = userdata.get(key)
            if token:
                break
        except Exception:
            pass

    if token:
        print(f"Push automatico in corso sul branch '{current_branch}'...")
        !git -C "{target_repo}" push https://{token}@github.com/Riccardo-27/DL-Quantum.git {current_branch}
        print(f"\n✔ Sincronizzazione completata con successo sul branch '{current_branch}'!")
    else:
        print("\n[INFO] Per abilitare il push automatico su Colab, configura il Secret 'GH_TOKEN' nell'icona 🔑.")
except ImportError:
    print(f"Ambiente locale: per il push esegui 'git push origin {current_branch}' da terminale.")
